# 02 — Modelo de referencia

Un baseline es un punto de comparación. DummyClassifier ignora las especificaciones;
la regresión logística aprende a distinguir cuatro categorías (aunque su nombre diga
«regresión», aquí es un clasificador). Solo usamos validación cruzada en entrenamiento.


In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "README.profe").exists():
    ROOT = ROOT.parent
if not (ROOT / "README.profe").exists():
    raise RuntimeError("Abra Jupyter desde la raíz del proyecto")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd

from src.data.dataset import load_data, split_data

df = load_data()
X_train, X_test, y_train, y_test = split_data(df)
print("Entrenamiento:", X_train.shape, "Test reservado:", X_test.shape)

Entrenamiento: (1600, 20) Test reservado: (400, 20)


In [2]:
from sklearn.model_selection import StratifiedKFold, cross_validate

from src.models.candidates import candidates

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
rows = []
for name in ["dummy", "logistic_regression"]:
    pipeline, _ = candidates()[name]
    scores = cross_validate(
        pipeline, X_train, y_train, cv=cv, scoring={"f1_macro": "f1_macro", "accuracy": "accuracy"}
    )
    rows.append(
        {
            "modelo": name,
            "F1 macro medio": scores["test_f1_macro"].mean(),
            "desviación F1": scores["test_f1_macro"].std(),
            "accuracy media": scores["test_accuracy"].mean(),
        }
    )
display(pd.DataFrame(rows).round(4))

,modelo,F1 macro medio,desviación F1,accuracy media
0,dummy,0.100,0.0000,0.2500
1,logistic_regression,0.948,0.0098,0.9481


## Interpretación

Accuracy es la fracción de aciertos. F1 combina precisión y recall; macro promedia
las cuatro clases con igual peso. Predecir siempre una clase da 25 % de accuracy en
datos equilibrados, pero F1 macro no tiene por qué ser 25 %.

El escalador se ajusta dentro de cada fold, nunca con todos los datos antes de validar.
No hemos calculado métricas de test en este notebook. En el siguiente se comparan
algoritmos e hiperparámetros y se documenta la selección final.
